In [ ]:
Template in pytorch nn function

def cross_entropy(
    input: Tensor,
    target: Tensor,
    weight: Tensor | None = None,
    size_average: bool | None = None,
    ignore_index: int = -100,
    reduce: bool | None = None,
    reduction: str = "mean",
    label_smoothing: float = 0.0,
) -> Tensor:
    r"""Compute the cross entropy loss between input logits and target.

    See :class:`~torch.nn.CrossEntropyLoss` for details.

    Args:
        input (Tensor) : Predicted unnormalized logits;
            see Shape section below for supported shapes.
        target (Tensor) : Ground truth class indices or class probabilities;
            see Shape section below for supported shapes.
        weight (Tensor, optional): a manual rescaling weight given to each
            class. If given, has to be a Tensor of size `C`
        size_average (bool, optional): Deprecated (see :attr:`reduction`).
        ignore_index (int, optional): Specifies a target value that is ignored
            and does not contribute to the input gradient. When :attr:`size_average` is
            ``True``, the loss is averaged over non-ignored targets. Note that
            :attr:`ignore_index` is only applicable when the target contains class indices.
            Default: -100
        reduce (bool, optional): Deprecated (see :attr:`reduction`).
        reduction (str, optional): Specifies the reduction to apply to the output:
            ``'none'`` | ``'mean'`` | ``'sum'``. ``'none'``: no reduction will be applied,
            ``'mean'``: the sum of the output will be divided by the number of
            elements in the output, ``'sum'``: the output will be summed. Note: :attr:`size_average`
            and :attr:`reduce` are in the process of being deprecated, and in the meantime,
            specifying either of those two args will override :attr:`reduction`. Default: ``'mean'``
        label_smoothing (float, optional): A float in [0.0, 1.0]. Specifies the amount
            of smoothing when computing the loss, where 0.0 means no smoothing. The targets
            become a mixture of the original ground truth and a uniform distribution as described in
            `Rethinking the Inception Architecture for Computer Vision <https://arxiv.org/abs/1512.00567>`__. Default: :math:`0.0`.

    Shape:
        - Input: Shape :math:`(C)`, :math:`(N, C)` or :math:`(N, C, d_1, d_2, ..., d_K)` with :math:`K \geq 1`
          in the case of `K`-dimensional loss.
        - Target: If containing class indices, shape :math:`()`, :math:`(N)` or :math:`(N, d_1, d_2, ..., d_K)` with
          :math:`K \geq 1` in the case of K-dimensional loss where each value should be between :math:`[0, C)`.
          If containing class probabilities, same shape as the input and each value should be between :math:`[0, 1]`.

        where:

        .. math::
            \begin{aligned}
                C ={} & \text{number of classes} \\
                N ={} & \text{batch size} \\
            \end{aligned}

    Examples::

        >>> # Example of target with class indices
        >>> input = torch.randn(3, 5, requires_grad=True)
        >>> target = torch.randint(5, (3,), dtype=torch.int64)
        >>> loss = F.cross_entropy(input, target)
        >>> loss.backward()
        >>>
        >>> # Example of target with class probabilities
        >>> input = torch.randn(3, 5, requires_grad=True)
        >>> target = torch.randn(3, 5).softmax(dim=1)
        >>> loss = F.cross_entropy(input, target)
        >>> loss.backward()
    """
    if has_torch_function_variadic(input, target, weight):
        return handle_torch_function(
            cross_entropy,
            (input, target, weight),
            input,
            target,
            weight=weight,
            size_average=size_average,
            ignore_index=ignore_index,
            reduce=reduce,
            reduction=reduction,
            label_smoothing=label_smoothing,
        )
    if size_average is not None or reduce is not None:
        reduction = _Reduction.legacy_get_string(size_average, reduce)
    return torch._C._nn.cross_entropy_loss(
        input,
        target,
        weight,
        # pyrefly: ignore [bad-argument-type]
        _Reduction.get_enum(reduction),
        ignore_index,
        label_smoothing,
    )


_IncompleteInputError: incomplete input (2391667427.py, line 167)

In [ ]:
import torch
from torch import Tensor

def function_cross_entropy(
    input: Tensor,   # [N, C]
    target: Tensor,  # [N]
    ignore_index: int = -100,
) -> Tensor:
    log_probs = input - torch.logsumexp(input, dim=1, keepdim=True)  # [N, C]

    valid_mask = target != ignore_index
    valid_target = target[valid_mask]                                # [M]
    valid_log_probs = log_probs[valid_mask]                          # [M, C]

    row_index = torch.arange(valid_target.size(0), device=input.device)
    nll = -valid_log_probs[row_index, valid_target]                  # [M]

    if nll.numel() == 0:
        return input.new_tensor(0.0)

    return nll.mean()

log_sum = torch.(input,dim=1)

In [ ]:
test case
import torch
import torch.nn.functional as F
input = torch.randn(3,5, requires_grad=True)
print(input.shape)
target = torch.randint(5, (3,), dtype=torch.int64)
print(target.shape)
loss = F.cross_entropy(input, target)
print(loss)
loss.backward()


torch.Size([3, 5])
torch.Size([3])
tensor(1.5934, grad_fn=<NllLossBackward0>)


In [11]:
import torch
import torch.nn.functional as F

input = torch.tensor([
    [2.1, 0.3, -1.2, 0.5, 1.0],
    [0.1, 1.7, 0.2, -0.8, 0.4],
    [1.2, 0.6, 0.1, 2.3, -0.5],
], requires_grad=True)

target = torch.tensor([0, 1, 3])

loss = F.cross_entropy(input, target)
print(loss)

print(F.softmax(input, dim=1))
loss.backward()
with torch.no_grad():
    input -= input.grad * 0.05
print(input)
print(input.grad)
print(F.softmax(input, dim=1))



tensor(0.5505, grad_fn=<NllLossBackward0>)
tensor([[0.5757, 0.0952, 0.0212, 0.1162, 0.1916],
        [0.1134, 0.5619, 0.1254, 0.0461, 0.1531],
        [0.1973, 0.1083, 0.0657, 0.5927, 0.0360]], grad_fn=<SoftmaxBackward0>)
tensor([[ 2.1071,  0.2984, -1.2004,  0.4981,  0.9968],
        [ 0.0981,  1.7073,  0.1979, -0.8008,  0.3974],
        [ 1.1967,  0.5982,  0.0989,  2.3068, -0.5006]], requires_grad=True)
tensor([[-0.1414,  0.0317,  0.0071,  0.0387,  0.0639],
        [ 0.0378, -0.1460,  0.0418,  0.0154,  0.0510],
        [ 0.0658,  0.0361,  0.0219, -0.1358,  0.0120]])
tensor([[0.5780, 0.0947, 0.0212, 0.1157, 0.1904],
        [0.1129, 0.5642, 0.1247, 0.0459, 0.1523],
        [0.1960, 0.1077, 0.0654, 0.5949, 0.0359]], grad_fn=<SoftmaxBackward0>)


In [14]:
from torch import Tensor
def function_cross_entropy(
    input: Tensor, #[N,c]
    target: Tensor,#[N]
    ignore_index: int = -100,
) -> Tensor:
    prob = torch.softmax(input, dim=1)  # [N, C]
    row_index = torch.arange(target.size(0))
    picked = prob[row_index,target]
    results= 0 - torch.log(picked) #
    return results.mean()

In [16]:
input = torch.tensor([
    [2.1, 0.3, -1.2, 0.5, 1.0],
    [0.1, 1.7, 0.2, -0.8, 0.4],
    [1.2, 0.6, 0.1, 2.3, -0.5],
], requires_grad=True)

target = torch.tensor([0, 1, 3])

loss = function_cross_entropy(input, target)
print(loss)

print(F.softmax(input, dim=1))
loss.backward()
with torch.no_grad():
    input -= input.grad * 0.05
print(input)
print(input.grad)
print(F.softmax(input, dim=1))

tensor(0.5505, grad_fn=<MeanBackward0>)
tensor([[0.5757, 0.0952, 0.0212, 0.1162, 0.1916],
        [0.1134, 0.5619, 0.1254, 0.0461, 0.1531],
        [0.1973, 0.1083, 0.0657, 0.5927, 0.0360]], grad_fn=<SoftmaxBackward0>)
tensor([[ 2.1071,  0.2984, -1.2004,  0.4981,  0.9968],
        [ 0.0981,  1.7073,  0.1979, -0.8008,  0.3974],
        [ 1.1967,  0.5982,  0.0989,  2.3068, -0.5006]], requires_grad=True)
tensor([[-0.1414,  0.0317,  0.0071,  0.0387,  0.0639],
        [ 0.0378, -0.1460,  0.0418,  0.0154,  0.0510],
        [ 0.0658,  0.0361,  0.0219, -0.1358,  0.0120]])
tensor([[0.5780, 0.0947, 0.0212, 0.1157, 0.1904],
        [0.1129, 0.5642, 0.1247, 0.0459, 0.1523],
        [0.1960, 0.1077, 0.0654, 0.5949, 0.0359]], grad_fn=<SoftmaxBackward0>)


In [ ]:
from torch import Tensor
def function_cross_entropy(
    input: Tensor, #[N,c]
    target: Tensor,#[N]
    ignore_index: int = -100,
) -> Tensor:
    valid_mask = target != ignore_index
    target = target[valid_mask]           
    input = input[valid_mask]            
    row_index = torch.arange(target.size(0))
    log_probs = torch.log_softmax(input, dim=1)  # [N, C]
    picked = log_probs[row_index, target]
    results = -picked

    return results.mean()

In [18]:
from torch import Tensor
def function_cross_entropy(
    input:Tensor, #[N,c]
    target:Tensor,#[N]
    ignore_index: int = -100,
) -> Tensor:
    valid_mask = target != ignore_index
    input = input[valid_mask]
    target = target[valid_mask]
    log_probs = torch.log_softmax(input,dim=1)
    row_index = torch.arange(target.size(0))
    selected_log_probs =log_probs[row_index, target]
    results = - selected_log_probs
    return results.mean()


In [19]:
input = torch.tensor([
    [2.1, 0.3, -1.2, 0.5, 1.0],
    [0.1, 1.7, 0.2, -0.8, 0.4],
    [1.2, 0.6, 0.1, 2.3, -0.5],
], requires_grad=True)

target = torch.tensor([0, 1, 3])

loss = function_cross_entropy(input, target)
print(loss)

print(F.softmax(input, dim=1))
loss.backward()
with torch.no_grad():
    input -= input.grad * 0.05
print(input)
print(input.grad)
print(F.softmax(input, dim=1))

tensor(0.5505, grad_fn=<MeanBackward0>)
tensor([[0.5757, 0.0952, 0.0212, 0.1162, 0.1916],
        [0.1134, 0.5619, 0.1254, 0.0461, 0.1531],
        [0.1973, 0.1083, 0.0657, 0.5927, 0.0360]], grad_fn=<SoftmaxBackward0>)
tensor([[ 2.1071,  0.2984, -1.2004,  0.4981,  0.9968],
        [ 0.0981,  1.7073,  0.1979, -0.8008,  0.3974],
        [ 1.1967,  0.5982,  0.0989,  2.3068, -0.5006]], requires_grad=True)
tensor([[-0.1414,  0.0317,  0.0071,  0.0387,  0.0639],
        [ 0.0378, -0.1460,  0.0418,  0.0154,  0.0510],
        [ 0.0658,  0.0361,  0.0219, -0.1358,  0.0120]])
tensor([[0.5780, 0.0947, 0.0212, 0.1157, 0.1904],
        [0.1129, 0.5642, 0.1247, 0.0459, 0.1523],
        [0.1960, 0.1077, 0.0654, 0.5949, 0.0359]], grad_fn=<SoftmaxBackward0>)


In [ ]:
from torch import Tensor
import torch

def function_cross_entropy(
    input:Tensor, #[N,c]
    target:Tensor,#[N]
    ignore_index: int = -100,
) -> Tensor:
    valid_mask = target !=ignore_index #[M]
    row_index = torch.arange(target.size(0))
    valid_input = input[valid_mask] #[M,C]
    valid_target = target[valid_mask] #[M]
    prob = torch.log_softmax(valid_input, dim=1) #shape [M,C] 必须沿着C类别维度积累
    selected_prob = prob[row_index, valid_target]
    loss = -selected_prob

    return loss.mean()
    



In [11]:
import torch.nn.functional as F
input = torch.tensor([
    [2.1, 0.3, -1.2, 0.5, 1.0],
    [0.1, 1.7, 0.2, -0.8, 0.4],
    [1.2, 0.6, 0.1, 2.3, -0.5],
], requires_grad=True)

target = torch.tensor([0, 1, 3])

loss = function_cross_entropy(input, target)
print(loss)

print(F.softmax(input, dim=1))
loss.backward()
with torch.no_grad():
    input -= input.grad * 0.05
print(input)
print(input.grad)
print(F.softmax(input, dim=1))


tensor(0.5505, grad_fn=<MeanBackward0>)
tensor([[0.5757, 0.0952, 0.0212, 0.1162, 0.1916],
        [0.1134, 0.5619, 0.1254, 0.0461, 0.1531],
        [0.1973, 0.1083, 0.0657, 0.5927, 0.0360]], grad_fn=<SoftmaxBackward0>)
tensor([[ 2.1071,  0.2984, -1.2004,  0.4981,  0.9968],
        [ 0.0981,  1.7073,  0.1979, -0.8008,  0.3974],
        [ 1.1967,  0.5982,  0.0989,  2.3068, -0.5006]], requires_grad=True)
tensor([[-0.1414,  0.0317,  0.0071,  0.0387,  0.0639],
        [ 0.0378, -0.1460,  0.0418,  0.0154,  0.0510],
        [ 0.0658,  0.0361,  0.0219, -0.1358,  0.0120]])
tensor([[0.5780, 0.0947, 0.0212, 0.1157, 0.1904],
        [0.1129, 0.5642, 0.1247, 0.0459, 0.1523],
        [0.1960, 0.1077, 0.0654, 0.5949, 0.0359]], grad_fn=<SoftmaxBackward0>)
